# 2강: LLM 프롬프트 엔지니어링으로 법률 조항 분류하기

1강에서 규칙 기반 분류기의 한계를 확인했습니다.
이번 강에서는 프롬프트 설계 전략으로 같은 문제를 더 잘 해결합니다.

## 학습 목표
1. Zero-Shot / One-Shot / Few-Shot 프롬프트 차이 이해
2. LLM 호출 흐름 이해 — 입력, 프롬프트 조합, 출력 파싱
3. Temperature 파라미터가 출력에 미치는 영향
4. 규칙 강화 프롬프트와 기본 프롬프트 성능 비교
5. Label-only 출력 단순화 전략

## 주의

이 노트북은 더미(Mock) 응답으로 동작합니다.
실제 API 키나 GPU 없이도 전체 흐름을 실행하고 확인할 수 있습니다.
실제 모델로 전환하려면 `mock_llm_call` 함수 내부만 교체하면 됩니다.
교체 방법은 마지막 섹션에 안내되어 있습니다.


## 0. 라이브러리 임포트

In [ ]:
import json
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(42)
print('라이브러리 로드 완료')


## 1. 평가 데이터 및 라벨 정의

In [ ]:
# 1-1. 평가 데이터셋 (1강과 동일 24건)

sample_data = [
    {'id':'D01','category':'DEF', 'text':'이 법은 국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 한다.'},
    {'id':'D02','category':'DEF', 'text':'이 법에서 사용하는 용어의 뜻은 다음 각 호와 같다.'},
    {'id':'D03','category':'DEF', 'text':'공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다.'},
    {'id':'D04','category':'DEF', 'text':'이 법은 대한민국 영역 안에서 이루어지는 정보 처리 행위에 적용한다.'},
    {'id':'R01','category':'RIGHT','text':'모든 국민은 법 앞에 평등하며 성별, 종교 또는 사회적 신분에 의하여 차별을 받지 아니한다.'},
    {'id':'R02','category':'RIGHT','text':'사업자는 이용자의 개인정보를 안전하게 관리하여야 한다.'},
    {'id':'R03','category':'RIGHT','text':'근로자는 안전하고 건강한 근무 환경에서 일할 권리를 가진다.'},
    {'id':'R04','category':'RIGHT','text':'누구든지 정당한 사유 없이 타인의 통신비밀을 침해하여서는 아니 된다.'},
    {'id':'P01','category':'PROC','text':'이 법을 위반한 자는 3년 이하의 징역 또는 3천만원 이하의 벌금에 처한다.'},
    {'id':'P02','category':'PROC','text':'신청인은 처분 통지를 받은 날부터 30일 이내에 이의신청을 할 수 있다.'},
    {'id':'P03','category':'PROC','text':'장관은 위반 사실을 조사한 후 청문 절차를 거쳐 등록을 취소할 수 있다.'},
    {'id':'P04','category':'PROC','text':'불법행위로 인한 손해배상 청구는 민사소송법에서 정한 절차에 따른다.'},
    {'id':'O01','category':'ORG', 'text':'분쟁 조정을 위하여 국무총리 소속으로 조정위원회를 둔다.'},
    {'id':'O02','category':'ORG', 'text':'위원회는 위원장 1명을 포함한 15명 이내의 위원으로 구성한다.'},
    {'id':'O03','category':'ORG', 'text':'법원은 사법권을 행사하며 대법원, 고등법원 및 지방법원으로 구성된다.'},
    {'id':'O04','category':'ORG', 'text':'중앙행정기관의 장은 소관 사무를 관장하고 소속 공무원을 지휘한다.'},
    {'id':'C01','category':'CRIT','text':'후보자는 선거일 현재 25세 이상인 국민이어야 한다.'},
    {'id':'C02','category':'CRIT','text':'지원 자격은 해당 분야 경력 3년 이상 및 학사 학위 이상으로 한다.'},
    {'id':'C03','category':'CRIT','text':'안전관리 기준은 시설 면적, 이용 인원 및 위험도에 따라 대통령령으로 정한다.'},
    {'id':'C04','category':'CRIT','text':'허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.'},
    {'id':'E01','category':'ETC', 'text':'이 법은 공포 후 6개월이 경과한 날부터 시행한다.'},
    {'id':'E02','category':'ETC', 'text':'이 법 시행 당시 종전의 규정에 따라 한 처분은 이 법에 따른 처분으로 본다.'},
    {'id':'E03','category':'ETC', 'text':'이 법의 시행에 필요한 사항은 대통령령으로 정한다.'},
    {'id':'E04','category':'ETC', 'text':'제3조의 개정규정은 이 법 시행 후 최초로 접수된 사건부터 적용한다.'},
]

label_desc = {
    'DEF':   '정의/목적/적용범위 조항',
    'RIGHT': '권리/의무/금지/책임 조항',
    'PROC':  '신청/심사/조사/불복/처벌 절차 조항',
    'ORG':   '기관/위원회/법원 등 조직의 설치/구성/권한 조항',
    'CRIT':  '자격/요건/기준/기간/수치 조건 조항',
    'ETC':   '시행일/경과조치/위임 등 기타 조항',
}

df = pd.DataFrame(sample_data)
ALLOWED     = list(label_desc.keys())
LABEL_GUIDE = '\n'.join(['- {}: {}'.format(k, v) for k, v in label_desc.items()])

print('평가 데이터: {} 건  /  {} 개 카테고리'.format(len(df), df['category'].nunique()))
print()
print('라벨 설명 (프롬프트 삽입용)')
print(LABEL_GUIDE)


## 2. 프롬프트 설계 — Zero / One / Few-Shot

프롬프트는 크게 세 요소로 구성됩니다.

| 구성 요소 | 역할 |
|-----------|------|
| System 프롬프트 | 모델 역할과 출력 형식을 고정합니다 |
| 라벨 설명 | 각 카테고리의 경계를 명시합니다 |
| 예시(Shot) | 분류 패턴을 학습하도록 유도합니다 |

Shot 수에 따른 차이:
- Zero-Shot: 예시 없이 라벨 설명만 제공
- One-Shot: 예시 1개 추가
- Few-Shot: 예시 3~5개 추가


In [ ]:
# 2-1. System / User 프롬프트 빌더

SYSTEM_PROMPT = (
    '너는 한국 법률 조항 분류기다. '
    '반드시 다음 6개 코드 중 하나만 선택하라: ' + ', '.join(ALLOWED) + '. '
    '응답은 반드시 JSON 한 줄로만 반환한다. '
    '형식: {"label": "DEF", "reason": "짧은 한국어 근거"}'
)

ONE_SHOT_EXAMPLE = {
    'text':   '신청인은 처분 통지를 받은 날부터 30일 이내에 이의신청을 할 수 있다.',
    'output': {'label': 'PROC', 'reason': '이의신청과 처리 절차를 규정한다'},
}

FEW_SHOT_EXAMPLES = [
    {'text':   '신청인은 처분 통지를 받은 날부터 30일 이내에 이의신청을 할 수 있다.',
     'output': {'label': 'PROC', 'reason': '신청/불복 절차 조항'}},
    {'text':   '근로자는 안전하고 건강한 근무 환경에서 일할 권리를 가진다.',
     'output': {'label': 'RIGHT', 'reason': '근로자의 권리 조항'}},
    {'text':   '분쟁 조정을 위하여 국무총리 소속으로 조정위원회를 둔다.',
     'output': {'label': 'ORG', 'reason': '위원회 설치 조항'}},
]


def build_user_prompt(text, mode='zero'):
    """
    법률 문장 분류 프롬프트를 생성합니다.

    Parameters
    ----------
    text : str  분류할 법률 문장
    mode : str  'zero' | 'one' | 'few'

    Returns
    -------
    str  완성된 User 프롬프트
    """
    lines = [
        '다음 법률 문장을 6개 코드 중 하나로 분류하라.',
        '라벨 설명:',
        LABEL_GUIDE,
    ]
    if mode == 'one':
        lines += [
            '',
            '[예시 1개]',
            '문장: ' + ONE_SHOT_EXAMPLE['text'],
            '정답: ' + json.dumps(ONE_SHOT_EXAMPLE['output'], ensure_ascii=False),
        ]
    elif mode == 'few':
        lines += ['', '[예시 {}개]'.format(len(FEW_SHOT_EXAMPLES))]
        for i, ex in enumerate(FEW_SHOT_EXAMPLES, 1):
            lines += [
                '예시 {}:'.format(i),
                '  문장: ' + ex['text'],
                '  정답: ' + json.dumps(ex['output'], ensure_ascii=False),
            ]
    lines += ['', '분류할 문장: ' + text, 'JSON으로만 응답하라.']
    return '\n'.join(lines)


# 프롬프트 미리보기
sample = '위원회는 위원장 1명을 포함한 15명 이내의 위원으로 구성한다.'
for mode in ['zero', 'one', 'few']:
    prompt = build_user_prompt(sample, mode=mode)
    print('[ {}-SHOT 프롬프트  총 {}자 ]'.format(mode.upper(), len(prompt)))
    print(prompt[:350])
    if len(prompt) > 350:
        print('...(이하 생략)')
    print()


## 3. Mock LLM — 더미 응답 시뮬레이터

실제 API나 GPU 없이 전체 실습 흐름을 확인하기 위한 더미 함수입니다.

`mock_llm_call`은 정답을 기반으로 현실적인 오답 패턴을 포함한 응답을 생성합니다.

| 조건 | 동작 |
|------|------|
| Zero-Shot | 정답률 약 75% |
| One-Shot  | 정답률 약 83% |
| Few-Shot  | 정답률 약 88% |
| temperature 높음 | 오답 확률 추가 상승 |
| 규칙 강화 프롬프트 | 경계 문장 정답률 개선 |

실제 모델로 바꾸려면 `mock_llm_call` 함수 내부만 교체하면 됩니다.


In [ ]:
# 3-1. Mock LLM 핵심 함수

# 실제 LLM이 자주 혼동하는 카테고리 쌍
CONFUSION_PAIRS = {
    'CRIT':  'ETC',
    'ETC':   'CRIT',
    'PROC':  'RIGHT',
    'RIGHT': 'PROC',
    'ORG':   'PROC',
}

# 모드별 기본 정답률
BASE_ACCURACY = {
    'zero':        0.75,
    'one':         0.83,
    'few':         0.88,
    'rule_aware':  0.92,
    'label_only':  0.90,
}

# 더미 근거 문장
DUMMY_REASONS = {
    'DEF':   '법의 목적 또는 용어 정의를 규정하는 조항',
    'RIGHT': '권리 또는 의무, 금지 행위를 규정하는 조항',
    'PROC':  '신청, 심사, 처벌 등 절차를 규정하는 조항',
    'ORG':   '위원회 또는 기관의 설치와 구성을 규정하는 조항',
    'CRIT':  '자격, 기준, 수치 요건을 규정하는 조항',
    'ETC':   '시행일, 경과조치, 위임 등 기타 행정 규정',
}


def mock_llm_call(text, true_label, mode='zero', temperature=0.0,
                   rule_aware=False, label_only=False):
    """
    실제 LLM 호출을 흉내 내는 더미 함수입니다.

    Parameters
    ----------
    text        : 분류할 법률 문장
    true_label  : 정답 라벨 (더미 정확도 시뮬레이션용)
    mode        : 'zero' | 'one' | 'few'
    temperature : 0.0이면 결정론적, 높을수록 오답 확률 증가
    rule_aware  : True면 경계 규칙 프롬프트 적용 (정답률 향상)
    label_only  : True면 reason 없이 label만 반환

    Returns
    -------
    dict : {'label': str, 'reason': str, 'raw_response': str}

    실제 모델로 교체하는 방법
    -------------------------
    이 함수 내부를 아래처럼 바꾸면 됩니다.

    [OpenAI]
        from openai import OpenAI
        client   = OpenAI(api_key='YOUR_KEY')
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': build_user_prompt(text, mode)},
            ]
        )
        raw    = response.choices[0].message.content
        parsed = extract_json(raw)
        return {'label': parsed['label'], 'reason': parsed.get('reason', ''), 'raw_response': raw}

    [HuggingFace]
        from transformers import AutoModelForCausalLM, AutoTokenizer
        model     = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-7B', device_map='auto')
        tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-7B')
        messages  = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': build_user_prompt(text, mode)},
        ]
        chat_text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs      = tokenizer([chat_text], return_tensors='pt').to(model.device)
        gen_ids     = model.generate(**inputs, max_new_tokens=128, do_sample=False)
        gen_ids     = [o[len(i):] for i, o in zip(inputs.input_ids, gen_ids)]
        raw         = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)[0]
        parsed      = extract_json(raw)
        return {'label': parsed['label'], 'reason': parsed.get('reason', ''), 'raw_response': raw}
    """
    acc_key  = 'rule_aware' if rule_aware else ('label_only' if label_only else mode)
    base_acc = BASE_ACCURACY.get(acc_key, 0.75)
    noise    = temperature * 0.15
    hit      = random.random() < (base_acc - noise)

    pred_label = true_label if hit else CONFUSION_PAIRS.get(true_label, random.choice(ALLOWED))
    reason     = DUMMY_REASONS[pred_label]

    if label_only:
        raw = json.dumps({'label': pred_label}, ensure_ascii=False)
    else:
        raw = json.dumps({'label': pred_label, 'reason': reason}, ensure_ascii=False)

    return {'label': pred_label, 'reason': reason, 'raw_response': raw}


def extract_json(text):
    """모델 응답에서 JSON 객체를 안전하게 추출합니다."""
    start, end = text.find('{'), text.rfind('}')
    if start == -1 or end == -1:
        return None
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None


# 동작 확인
test = {'text': '이 법은 공포 후 6개월이 경과한 날부터 시행한다.', 'true': 'ETC'}
print('Mock LLM 동작 확인')
print('문장:', test['text'])
print('정답:', test['true'])
print()
for m in ['zero', 'one', 'few']:
    r  = mock_llm_call(test['text'], test['true'], mode=m)
    ok = '정답' if r['label'] == test['true'] else '오답'
    print('  {:4s}  ->  {}  ({})  raw: {}'.format(m, r['label'], ok, r['raw_response']))
print()
print('seed=42로 고정했으므로 매번 같은 결과가 나옵니다.')


## 4. Zero-Shot vs One-Shot — 전체 데이터 비교

24건 전체에 대해 두 모드의 정확도와 오답 패턴을 비교합니다.


In [ ]:
# 4-1. Zero-Shot vs One-Shot 전체 평가

results = []
for _, row in df.iterrows():
    for mode in ['zero', 'one']:
        r = mock_llm_call(row['text'], row['category'], mode=mode)
        results.append({
            'id':      row['id'],
            'true':    row['category'],
            'mode':    mode,
            'pred':    r['label'],
            'correct': r['label'] == row['category'],
            'reason':  r['reason'],
        })

res_df = pd.DataFrame(results)

print('Zero-Shot vs One-Shot 정확도')
summary = res_df.groupby('mode')['correct'].mean().rename('accuracy').round(3)
print(summary.to_frame())
print()

print('카테고리별 정확도')
cat_acc = res_df.groupby(['mode', 'true'])['correct'].mean().unstack('mode').round(2)
print(cat_acc)
print()

errors = res_df[(res_df['mode'] == 'one') & (~res_df['correct'])]
print('One-Shot 오답 {} 건'.format(len(errors)))
if not errors.empty:
    print(errors[['id', 'true', 'pred', 'reason']].to_string(index=False))


## 5. Temperature 파라미터 실험

같은 프롬프트, 다른 Temperature 값으로 출력 안정성 차이를 확인합니다.

Temperature는 모델이 다음 토큰을 선택할 때 확률 분포를 얼마나 평탄하게 만들지 조절합니다.

| 설정 | temperature | 특징 |
|------|------------|------|
| deterministic | 0.0 | 매번 동일한 출력, 가장 안정적 |
| balanced | 0.3 | 실무 권장, 약간의 다양성 허용 |
| creative | 0.8 | 다양한 출력, 분류에서는 불안정 |


In [ ]:
# 5-1. Temperature별 정확도 비교

EXPERIMENTS = [
    {'name': 'deterministic', 'temperature': 0.0, 'desc': 'Greedy — 안정적'},
    {'name': 'balanced',      'temperature': 0.3, 'desc': '실무 권장'},
    {'name': 'creative',      'temperature': 0.8, 'desc': '다양성 허용'},
]

EVAL_SAMPLES = [
    {'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.',
     'true': 'CRIT', 'note': '수치 요건 조항'},
    {'text': '이 법은 공포 후 6개월이 경과한 날부터 시행한다.',
     'true': 'ETC',  'note': '시행일 조항'},
    {'text': '공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다.',
     'true': 'DEF',  'note': '정의 조항'},
    {'text': '누구든지 정당한 사유 없이 타인의 통신비밀을 침해하여서는 아니 된다.',
     'true': 'RIGHT','note': '금지 조항'},
    {'text': '장관은 위반 사실을 조사한 후 청문 절차를 거쳐 등록을 취소할 수 있다.',
     'true': 'PROC', 'note': '처분 절차 조항'},
]

temp_rows = []
for exp in EXPERIMENTS:
    correct = 0
    for s in EVAL_SAMPLES:
        r  = mock_llm_call(s['text'], s['true'], mode='zero', temperature=exp['temperature'])
        ok = r['label'] == s['true']
        correct += int(ok)
        temp_rows.append({
            'setting':     exp['name'],
            'temperature': exp['temperature'],
            'note':        s['note'],
            'true':        s['true'],
            'pred':        r['label'],
            'correct':     ok,
        })
    print('[{:14s}  temp={}]  정확도: {}/{}  ({})'.format(
        exp['name'], exp['temperature'], correct, len(EVAL_SAMPLES), exp['desc']))

temp_df = pd.DataFrame(temp_rows)

acc_by_temp = temp_df.groupby(['setting', 'temperature'])['correct'].mean().reset_index()
acc_by_temp.columns = ['setting', 'temperature', 'accuracy']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(acc_by_temp['setting'], acc_by_temp['accuracy'],
              color=['forestgreen', 'steelblue', 'firebrick'],
              edgecolor='black', width=0.5)
for bar, row in zip(bars, acc_by_temp.itertuples()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            '{:.0%}'.format(row.accuracy), ha='center', va='bottom', fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy by Temperature Setting')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('분류 과제에서는 temperature=0(결정론적)이 가장 안정적입니다.')


## 6. Few-Shot 예시 개수 실험

예시를 0개, 1개, 3개, 5개로 늘렸을 때 정확도 변화를 확인합니다.

- 0 -> 1개 구간에서 가장 큰 향상이 기대됩니다.
- 3 -> 5개 구간에서는 향상이 작거나 오히려 하락할 수 있습니다.
- 예시가 특정 카테고리에 편향되면 모델도 그 방향으로 치우칩니다.
- 예시의 양보다 질이 더 중요합니다.


In [ ]:
# 6-1. Few-Shot 개수별 성능 비교

EXTRA_EXAMPLES = [
    {'text':   '이 법 시행에 필요한 사항은 대통령령으로 정한다.',
     'output': {'label': 'ETC', 'reason': '시행 위임 조항'}},
    {'text':   '모든 국민은 법 앞에 평등하며 차별을 받지 아니한다.',
     'output': {'label': 'RIGHT', 'reason': '평등권 조항'}},
]

SHOT_CONFIG = {
    0: {'mode': 'zero'},
    1: {'mode': 'one'},
    3: {'mode': 'few'},
    5: {'mode': 'few'},
}

# 5개 예시는 편향 효과로 3개보다 낮게 시뮬레이션
MOCK_ACC_OVERRIDE = {5: 0.84}

fewshot_rows = []
for n_ex, cfg in SHOT_CONFIG.items():
    correct = 0
    for s in EVAL_SAMPLES:
        override = MOCK_ACC_OVERRIDE.get(n_ex)
        if override is not None:
            hit  = random.random() < override
            pred = s['true'] if hit else ({'CRIT':'ETC','ETC':'CRIT'}.get(s['true'], ALLOWED[0]))
            r    = {'label': pred, 'reason': ''}
        else:
            r = mock_llm_call(s['text'], s['true'], mode=cfg['mode'])
        ok = r['label'] == s['true']
        correct += int(ok)
        fewshot_rows.append({'n_examples': n_ex, 'true': s['true'],
                             'pred': r['label'], 'correct': ok})
    print('예시 {}개  ->  정확도: {}/{}'.format(n_ex, correct, len(EVAL_SAMPLES)))

fs_df      = pd.DataFrame(fewshot_rows)
fs_summary = fs_df.groupby('n_examples')['correct'].mean().rename('accuracy').round(3)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(fs_summary.index, fs_summary.values, 'o-', color='steelblue', linewidth=2, markersize=8)
for x, y in zip(fs_summary.index, fs_summary.values):
    ax.annotate('{:.0%}'.format(y), (x, y),
                textcoords='offset points', xytext=(0, 10), ha='center', fontsize=11)
ax.set_xticks([0, 1, 3, 5])
ax.set_xlabel('Number of Few-Shot Examples')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy by Number of Few-Shot Examples')
ax.set_ylim(0.5, 1.05)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('요약')
print(fs_summary.to_frame())
print()
print('5개 예시가 3개보다 오히려 낮아질 수 있습니다. 예시의 질이 더 중요합니다.')


## 7. 규칙 강화 프롬프트 전략

예시 개수를 늘리는 것보다 효과적인 방법이 있습니다.
헷갈리는 라벨 경계를 직접 문장으로 명시하는 것입니다.

```
기본 프롬프트      : 라벨 설명 + 문장
규칙 강화 프롬프트 : 라벨 설명 + 경계 규칙 + 문장
```

핵심 경계 규칙 예시:
- CRIT vs ETC: 숫자나 요건이 있어도 시행일, 경과조치면 ETC
- PROC vs ETC: 절차 자체를 직접 설명하면 PROC, 시행 시기나 부칙이면 ETC


In [ ]:
# 7-1. 기본 vs 규칙 강화 프롬프트 성능 비교

RULES_FOR_CONFUSION = [
    'DEF는 정의, 목적, 적용범위처럼 개념을 설명하는 문장이다.',
    'RIGHT는 권리, 의무, 금지, 책임처럼 행위의 허용과 금지를 다루는 문장이다.',
    'PROC는 신청, 심사, 조사, 불복, 처벌처럼 절차나 처리 흐름을 다루는 문장이다.',
    'ORG는 위원회, 법원, 기관, 조직의 설치, 구성, 권한을 다루는 문장이다.',
    'CRIT는 자격, 요건, 기준, 기간, 수치 조건처럼 충족해야 할 조건을 다루는 문장이다.',
    'ETC는 시행일, 경과조치, 위임, 부칙처럼 나머지 행정적 규정을 다루는 문장이다.',
    '주의: 숫자나 요건이 있어도 시행일과 경과조치는 ETC로 판단한다.',
    '주의: 절차 자체를 직접 설명하면 PROC, 시행 시기나 부칙이면 ETC로 판단한다.',
]


def build_rule_aware_prompt(text):
    """
    경계 규칙이 추가된 프롬프트를 생성합니다.

    기본 프롬프트 대비 추가되는 것:
        '경계 규칙:' 섹션 (RULES_FOR_CONFUSION 리스트)
    """
    lines = [
        '다음 법률 문장을 6개 코드 중 하나로 분류하라.',
        '라벨 설명:', LABEL_GUIDE,
        '', '경계 규칙:',
    ]
    lines.extend(['- ' + r for r in RULES_FOR_CONFUSION])
    lines += [
        '', '분류할 문장: ' + text,
        '반드시 다음 JSON 형식으로만 답하라.',
        '{"label": "DEF|RIGHT|PROC|ORG|CRIT|ETC", "reason": "짧은 근거"}',
    ]
    return '\n'.join(lines)


RULE_EVAL = [
    {'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.',
     'true': 'CRIT', 'note': '수치 요건'},
    {'text': '이 법은 공포 후 6개월이 경과한 날부터 시행한다.',
     'true': 'ETC',  'note': '시행일'},
    {'text': '공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다.',
     'true': 'DEF',  'note': '정의'},
    {'text': '누구든지 정당한 사유 없이 타인의 통신비밀을 침해하여서는 아니 된다.',
     'true': 'RIGHT','note': '금지 규정'},
    {'text': '장관은 위반 사실을 조사한 후 청문 절차를 거쳐 등록을 취소할 수 있다.',
     'true': 'PROC', 'note': '처분 절차'},
]

strategy_rows = []
print('{:<20} {:<6} {:<12} {:<12} 개선'.format('문장', '정답', 'baseline', 'rule_aware'))
print('-' * 60)

for s in RULE_EVAL:
    b = mock_llm_call(s['text'], s['true'], mode='zero')
    r = mock_llm_call(s['text'], s['true'], mode='zero', rule_aware=True)
    b_ok = b['label'] == s['true']
    r_ok = r['label'] == s['true']
    improved = '향상' if (r_ok and not b_ok) else ('하락' if (not r_ok and b_ok) else '-')
    print('{:<20} {:<6} {} ({:<8}) {} ({:<8}) {}'.format(
        s['note'], s['true'],
        b['label'], '정답' if b_ok else '오답',
        r['label'], '정답' if r_ok else '오답',
        improved))
    strategy_rows.append({'strategy': 'baseline',   'true': s['true'], 'pred': b['label'], 'correct': b_ok})
    strategy_rows.append({'strategy': 'rule_aware',  'true': s['true'], 'pred': r['label'], 'correct': r_ok})

strat_df = pd.DataFrame(strategy_rows)
print()
print('전략별 정확도')
print(strat_df.groupby('strategy')['correct'].mean().round(3).rename('accuracy').to_frame())
print()
print('규칙 강화 프롬프트 미리보기 (시행일 문장)')
print('-' * 60)
print(build_rule_aware_prompt(RULE_EVAL[1]['text']))


## 8. 출력 단순화 전략 — Label Only

모델에게 label과 reason을 동시에 생성하도록 요구하면 두 가지 문제가 생깁니다.

1. JSON 형식이 깨질 수 있습니다.
2. reason 생성에 집중하다 label을 틀릴 수 있습니다.

전략: reason을 제거하고 label만 반환하도록 강제합니다.

```json
기존     : {"label": "CRIT", "reason": "자본금 요건을 규정"}
단순화   : {"label": "CRIT"}
```

| 방법 | 장점 | 단점 |
|------|------|------|
| label + reason | 근거 확인 가능, 오답 분석 용이 | JSON 파싱 실패 위험 |
| label only | 안정적, 빠름 | 근거 없음 |


In [ ]:
# 8-1. Label-Only vs Label+Reason 비교

def build_label_only_prompt(text):
    """
    label만 출력하도록 요구하는 프롬프트를 생성합니다.

    기본 프롬프트 대비 변경:
        - reason 생성 금지 규칙 추가
        - 출력 형식을 {"label": "..."} 으로 단순화
    """
    lines = [
        '다음 법률 문장을 6개 코드 중 하나로 분류하라.',
        '라벨 설명:', LABEL_GUIDE,
        '',
        '규칙:',
        'reason은 쓰지 말고 label만 JSON으로 반환하라.',
        '반드시 다음 형식만 출력하라: {"label": "DEF|RIGHT|PROC|ORG|CRIT|ETC"}',
        '',
        '분류할 문장: ' + text,
    ]
    return '\n'.join(lines)


label_rows = []
print('{:<20} {:<6} {:<14} {}'.format('문장', '정답', 'label+reason', 'label_only'))
print('-' * 55)

for s in RULE_EVAL:
    b  = mock_llm_call(s['text'], s['true'], mode='zero')
    lo = mock_llm_call(s['text'], s['true'], mode='zero', label_only=True)
    b_ok  = b['label']  == s['true']
    lo_ok = lo['label'] == s['true']
    print('{:<20} {:<6} {} ({:<8}) {} ({})'.format(
        s['note'], s['true'],
        b['label'],  '정답' if b_ok  else '오답',
        lo['label'], '정답' if lo_ok else '오답'))
    label_rows.append({'strategy': 'label+reason', 'true': s['true'], 'pred': b['label'],  'correct': b_ok})
    label_rows.append({'strategy': 'label_only',   'true': s['true'], 'pred': lo['label'], 'correct': lo_ok})

lo_df = pd.DataFrame(label_rows)
print()
print('출력 형식별 정확도')
print(lo_df.groupby('strategy')['correct'].mean().round(3).rename('accuracy').to_frame())

print()
print('응답 형식 비교')
sample = RULE_EVAL[0]
b_raw  = mock_llm_call(sample['text'], sample['true'], mode='zero')['raw_response']
lo_raw = mock_llm_call(sample['text'], sample['true'], mode='zero', label_only=True)['raw_response']
print('label+reason :', b_raw)
print('label_only   :', lo_raw)
print('글자 수: label+reason={}자 / label_only={}자'.format(len(b_raw), len(lo_raw)))


## 9. 전체 전략 종합 비교

In [ ]:
# 9-1. 모든 전략을 한 번에 비교하는 시각화

strategy_summary = pd.DataFrame([
    {'strategy': 'Rule-based\n(1강 기준선)', 'accuracy': 0.79, 'group': 'baseline'},
    {'strategy': 'Zero-Shot',                'accuracy': 0.75, 'group': 'llm'},
    {'strategy': 'One-Shot',                 'accuracy': 0.83, 'group': 'llm'},
    {'strategy': 'Few-Shot (3)',              'accuracy': 0.88, 'group': 'llm'},
    {'strategy': 'Rule-Aware\nPrompt',        'accuracy': 0.92, 'group': 'advanced'},
    {'strategy': 'Label-Only\n+ Rule-Aware',  'accuracy': 0.90, 'group': 'advanced'},
])

color_map = {'baseline': 'lightsteelblue', 'llm': 'steelblue', 'advanced': 'firebrick'}
bar_colors = [color_map[g] for g in strategy_summary['group']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(strategy_summary['strategy'], strategy_summary['accuracy'],
              color=bar_colors, edgecolor='black', width=0.6)
for bar, acc in zip(bars, strategy_summary['accuracy']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            '{:.0%}'.format(acc), ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylim(0.5, 1.05)
ax.set_ylabel('Accuracy')
ax.set_title('Classification Strategy Comparison (Mock Results)')
ax.axhline(0.79, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Rule-based baseline')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('전략 요약')
print('-' * 50)
for _, row in strategy_summary.iterrows():
    name = row['strategy'].replace('\n', ' ')
    print('  {:<28}  ->  {:.0%}'.format(name, row['accuracy']))
print('-' * 50)
print()
print('핵심 인사이트')
print('  1. Zero-Shot LLM이 규칙 기반보다 낮을 수도 있습니다.')
print('  2. One-Shot 하나만 추가해도 큰 폭의 향상이 가능합니다.')
print('  3. 규칙 강화 프롬프트가 예시 추가보다 효과적인 경우가 많습니다.')
print('  4. 오답 분석 -> 규칙 추가 -> 재평가 사이클이 실무에서 가장 효과적입니다.')


## 10. 종합 정리

| 실험 | 핵심 발견 |
|------|-----------|
| Zero vs One-Shot | 예시 1개만 추가해도 의미 있는 성능 향상 |
| Few-Shot 개수 | 5개 이상에서 편향 위험, 예시의 질이 더 중요 |
| Temperature | 분류 과제는 낮은 값(0~0.3)이 안정적 |
| 규칙 강화 | 경계 명시가 예시 추가보다 효과적 |
| Label-Only | 작은 모델에서 JSON 파싱 안정성 향상 |

## 실제 모델로 전환하는 방법

`mock_llm_call` 함수 내부를 아래 코드로 교체하면 됩니다.
프롬프트 빌더, 평가 로직, 시각화 코드는 그대로 재사용됩니다.

```python
# OpenAI 사용 시
from openai import OpenAI
client   = OpenAI(api_key='YOUR_KEY')
response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': build_user_prompt(text, mode)},
    ]
)
raw    = response.choices[0].message.content
parsed = extract_json(raw)
return {'label': parsed['label'], 'reason': parsed.get('reason', ''), 'raw_response': raw}

# HuggingFace 사용 시
from transformers import AutoModelForCausalLM, AutoTokenizer
model     = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-7B', device_map='auto')
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-7B')
messages  = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user',   'content': build_user_prompt(text, mode)},
]
chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs    = tokenizer([chat_text], return_tensors='pt').to(model.device)
gen_ids   = model.generate(**inputs, max_new_tokens=128, do_sample=False)
gen_ids   = [o[len(i):] for i, o in zip(inputs.input_ids, gen_ids)]
raw       = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)[0]
parsed    = extract_json(raw)
return {'label': parsed['label'], 'reason': parsed.get('reason', ''), 'raw_response': raw}
```
